# 아파트 매매 데이터 수집

In [3]:
# 필요 모듈 import
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException, NoSuchElementException, StaleElementReferenceException
import csv, time, re
import pandas as pd
from datetime import datetime, timedelta
import os 
import glob


In [8]:
download_dir = os.path.abspath("../data/raw/apt_sale")

# ✅ 2. Chrome 옵션 설정
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")  # 헤드리스에서도 레이아웃 강제 렌더링

prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)

# ✅ 3. 드라이버 실행
service = Service(executable_path="../chromedriver-mac-arm64/chromedriver")
driver = webdriver.Chrome(service=service, options=options)

In [9]:
driver.get('https://rt.molit.go.kr/pt/xls/xls.do?mobileAt=')
wait = WebDriverWait(driver, 5)


In [10]:
# 시작 날짜 (예: 2022-06-02)
base_date = datetime.strptime("2020-07-11", "%Y-%m-%d")
end_date = datetime.strptime("2025-06-20", "%Y-%m-%d")

start_date_list = []
end_date_list = []

while base_date < end_date:
    start = base_date
    end = min(base_date + timedelta(days=365), end_date)
    
    start_date_list.append(start.strftime("%Y-%m-%d"))
    end_date_list.append(end.strftime("%Y-%m-%d"))
    
    base_date = end + timedelta(days=1)

print(list(zip(start_date_list, end_date_list)))

[('2020-07-11', '2021-07-11'), ('2021-07-12', '2022-07-12'), ('2022-07-13', '2023-07-13'), ('2023-07-14', '2024-07-13'), ('2024-07-14', '2025-06-20')]


In [12]:
for i in range(len(start_date_list)):
    # 날짜 선택 및 입력
    start_date = driver.find_element(By.ID, "srhFromDt")
    end_date = driver.find_element(By.ID, "srhToDt")
    start_date.send_keys(start_date_list[i])
    end_date.send_keys(end_date_list[i])
    
    #시도 선택 및 입력
    sido = driver.find_element(By.ID, "srhSidoCd")
    sido = Select(sido)
    sido.select_by_value("11000")
    
    # Download Btn Click
    download_btn = driver.find_element(By.XPATH, "//button[contains(text(), 'CSV 다운')]")
    download_btn.click()
    print(f"{start_date_list[i]} ~ {end_date_list[i]} 까지 매매데이터 다운 완료")
    time.sleep(5)

print("매매데이터를 전부 다운 했습니다.")

2020-07-11 ~ 2021-07-11 까지 매매데이터 다운 완료
2021-07-12 ~ 2022-07-12 까지 매매데이터 다운 완료
2022-07-13 ~ 2023-07-13 까지 매매데이터 다운 완료
2023-07-14 ~ 2024-07-13 까지 매매데이터 다운 완료
2024-07-14 ~ 2025-06-20 까지 매매데이터 다운 완료
매매데이터를 전부 다운 했습니다.


In [13]:
driver.quit()

# CSV 전처리

In [4]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일',
    '거래유형', '중개사소재지', '등기일자'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../data/raw/apt_sale"

# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")

# 데이터프레임 병합
merged_df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(merged_df)}건")

읽기 완료: 아파트(매매)_실거래가_20250620103841.csv
읽기 완료: 아파트(매매)_실거래가_20250620103317.csv
읽기 완료: 아파트(매매)_실거래가_20250620103854.csv
읽기 완료: 아파트(매매)_실거래가_20250620103311.csv
읽기 완료: 아파트(매매)_실거래가_20250620103846.csv
읽기 완료: 아파트(매매)_실거래가_20250620103404.csv
읽기 완료: 아파트(매매)_실거래가_20250620103822.csv
읽기 완료: 아파트(매매)_실거래가_20250620132033.csv
읽기 완료: 아파트(매매)_실거래가_20250620132025.csv
읽기 완료: 아파트(매매)_실거래가_20250620132018.csv
읽기 완료: 아파트(매매)_실거래가_20250620103400.csv
읽기 완료: 아파트(매매)_실거래가_20250620132012.csv
읽기 완료: 아파트(매매)_실거래가_20250620100439.csv

 병합 완료: 총 512361건


In [5]:
merged_df['계약년월'].unique()

array([202109, 202108, 202107, 202106, 202105, 202104, 202103, 202102,
       202101, 202012, 202011, 202010, 202009, 202406, 202405, 202404,
       202403, 202402, 202401, 202312, 202311, 202310, 202309, 202308,
       202307, 202306, 202305, 202304, 202303, 202302, 202301, 202212,
       202211, 202210, 202209, 202208, 202207, 202206, 202205, 202204,
       202203, 202202, 202201, 202112, 202111, 202110, 202407, 202008,
       202007, 202506, 202505, 202504, 202503, 202502, 202501, 202412,
       202411, 202410, 202409, 202408])

In [6]:
merged_df.columns

Index(['시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일', '거래금액(만원)',
       '동', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형', '중개사소재지',
       '등기일자'],
      dtype='object')

In [7]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 512361 entries, 0 to 512360
Data columns (total 19 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   시군구       512361 non-null  object 
 1   번지        512361 non-null  object 
 2   본번        512361 non-null  int64  
 3   부번        512361 non-null  int64  
 4   단지명       512361 non-null  object 
 5   전용면적(㎡)   512361 non-null  float64
 6   계약년월      512361 non-null  int64  
 7   계약일       512361 non-null  int64  
 8   거래금액(만원)  512361 non-null  object 
 9   동         512361 non-null  object 
 10  층         512361 non-null  int64  
 11  매수자       512361 non-null  object 
 12  매도자       512361 non-null  object 
 13  건축년도      512361 non-null  int64  
 14  도로명       512361 non-null  object 
 15  해제사유발생일   512361 non-null  object 
 16  거래유형      512361 non-null  object 
 17  중개사소재지    512361 non-null  object 
 18  등기일자      512361 non-null  object 
dtypes: float64(1), int64(6), object(12)
memory u

In [8]:
merged_df.head(1)

,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,서울특별시 중랑구 면목동,193-1,193,1,면목한신,35.3,202109,1,"40,000",-,5,-,-,1988,중랑천로 20,-,-,-,-


In [11]:
len(merged_df)

512361

In [11]:
import pandas as pd
import pymysql

def get_connection():
    return pymysql.connect(
        host='localhost',
        user='root',
        password='As589788@@',
        db='apt_price',
        charset='utf8mb4',
        cursorclass=pymysql.cursors.DictCursor
    )

# CSV 읽기
df = merged_df

# 데이터 전처리 (예: 거래금액 쉼표 제거)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(str)

# NULL 허용 컬럼의 '-' 처리 (예: '동', '매수자' 등)
df.replace('-', None, inplace=True)

# 컬럼명 영문 매핑
df = df.rename(columns={
    '시군구': 'region',
    '번지': 'lot_number',
    '본번': 'main_number',
    '부번': 'sub_number',
    '단지명': 'complex_name',
    '전용면적(㎡)': 'area_m2',
    '계약년월': 'contract_ym',
    '계약일': 'contract_day',
    '거래금액(만원)': 'price_str',
    '동': 'building_name',
    '층': 'floor',
    '매수자': 'buyer',
    '매도자': 'seller',
    '건축년도': 'built_year',
    '도로명': 'street_name',
    '해제사유발생일': 'cancel_date',
    '거래유형': 'deal_type',
    '중개사소재지': 'agency_location',
    '등기일자': 'registration_date'
})

# DB 삽입
conn = get_connection()
cursor = conn.cursor()

sql = """
INSERT INTO apt_raw_sales (
    region, lot_number, main_number, sub_number, complex_name, area_m2,
    contract_ym, contract_day, price_str, building_name, floor, buyer,
    seller, built_year, street_name, cancel_date, deal_type, agency_location, registration_date
) VALUES (
    %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s
)
"""

data_tuples = [tuple(row) for row in df.to_numpy()]

try:
    cursor.executemany(sql, data_tuples)
    conn.commit()
finally:
    cursor.close()
    conn.close()

In [15]:
df = pd.read_csv("/Users/kim-youngho/git/AptPrice-SatelliteSentiment/data/interim/apt/apt_remove_duplicated.csv")
df.head()

,area_m2,complex_name,floor,contract_day,street_name,built_year,price_per_m2,apartment_age,alpha
0,84.86,태솔에버빌,3,2020-07-11,삼양로80길 55,2008,424.228140,12,0.600000
1,83.19,광남캐스빌3,2,2020-07-11,성내로6가길 98,2004,717.634331,16,0.466667
2,83.40,신림동이모르젠,2,2020-07-11,신사로 84,2003,713.429257,17,0.433333
3,82.68,현대강변,10,2020-07-11,자양로3가길 43,1991,1100.628931,29,0.033333
4,84.85,현대,8,2020-07-11,토정로 242,1996,1060.695345,24,0.200000
